# Sentiment job: verification tests

Run top to bottom on a GPU instance (ml.g6 / L4), from the `notebooks/` folder, after a kernel restart.
Each section prints `PASS` / `FAIL` lines; the last cell prints a summary.

| Section | What it checks | GPU | Writes |
|---|---|---|---|
| 1 | Config, secrets, Redshift read | no | nothing |
| 2 | Follow-ups extraction | no | nothing |
| 3 | Tokenizer: single `<bos>`, prompt lengths, batch plan | no | nothing |
| 4 | Old vs new scoring on a sample (quality, time, memory) | yes, 2 model loads | nothing |
| 5 | Full production scoring + output shape | yes, 1 model load | nothing |
| 6 | COPY into a scratch table (created and dropped here) | no | scratch table only |

Production tables are never touched.

## 0. Setup

In [ ]:
import gc
import logging
import math
import os
import sys
import time
from pathlib import Path

# Absolute paths, so spawned child processes (section 4) see them too
sys.path.insert(0, str(Path("../src").resolve()))
sys.path.insert(0, os.getcwd())
# Less fragmentation between big allocations; child processes inherit it
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(name)s - %(message)s", force=True)

import pandas as pd
import torch
from transformers import AutoTokenizer

from asksa_follow_up.ask_sa_followup import FEW_SHOTS, SYSTEM_RULES, FollowupLLM
from asksa_jobs import sentiment_job as sj
from asksa_jobs.common import (
    export_hf_token,
    fetch_df,
    load_df_to_redshift,
    load_settings,
    read_query,
    redshift_connect,
)

RESULTS = []


def check(name, ok, details=""):
    """Record and print one test result."""
    status = "PASS" if ok else "FAIL"
    RESULTS.append((status, name, details))
    print(f"{status} | {name}" + (f" | {details}" if details else ""))


def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


def peak_gb():
    return torch.cuda.max_memory_allocated() / 1024**3


def check_gpu_free(before_what, min_free_gb=20):
    """The 9B model needs ~17 GB; less free memory means CPU offload or OOM."""
    free_gb = torch.cuda.mem_get_info()[0] / 1024**3
    check(f"GPU free before {before_what}", free_gb >= min_free_gb, f"{free_gb:.1f} GB free")
    if free_gb < min_free_gb:
        raise RuntimeError("GPU memory is still in use: restart the kernel and check !nvidia-smi")


check("CUDA available", torch.cuda.is_available())
GPU_TOTAL_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {torch.cuda.get_device_name(0)}, {GPU_TOTAL_GB:.1f} GB, bf16 supported: {torch.cuda.is_bf16_supported()}")

## 1. Config, secrets, Redshift read

In [ ]:
settings = load_settings()
params = sj.SentimentParams()
print(settings)
print(params)

token = export_hf_token(settings)
check("HF token loaded from Secrets Manager", token.startswith("hf_"), f"{token[:5]}...")

tickers_df = fetch_df(settings, read_query(settings, "get_tickers.sql"))
check("Redshift read works", len(tickers_df) > 0, f"{len(tickers_df)} tickers")

## 2. Follow-ups extraction

In [ ]:
followups = sj.load_followups(settings).reset_index(drop=True)
check("there are follow-ups to test on", len(followups) > 0, f"{len(followups)} rows")

needed = {"user_id", "chat_id", "followup_number", "followup_time", "initial_query", "ask_sa_answer", "followup_query"}
check("follow-ups have expected columns", needed <= set(followups.columns), str(sorted(needed - set(followups.columns))))
followups.head()

## 3. Tokenizer checks (no model)

Uses the production prompt code (`_safe_pack`, `_render`, `_tokenize`, `_make_batches`) with the tokenizer only.

In [ ]:
def make_tokenizer_only_llm(params, token):
    """FollowupLLM without the model: same prompt/tokenization code as production."""
    llm = FollowupLLM.__new__(FollowupLLM)
    llm.model_id = params.model_id
    llm.max_input_tokens = params.max_input_tokens
    llm.batch_size = params.batch_size
    llm.max_batch_tokens = params.max_batch_tokens
    llm.system_rules = SYSTEM_RULES
    llm.few_shots = FEW_SHOTS
    llm.max_answer_chars = 4000
    llm.tokenizer = AutoTokenizer.from_pretrained(params.model_id, token=token)
    llm.tokenizer.padding_side = "left"
    return llm


tok_llm = make_tokenizer_only_llm(params, token)

# 3.1 single <bos>
ids = tok_llm._tokenize(tok_llm._render(["hi"]))["input_ids"][0]
first_tokens = tok_llm.tokenizer.convert_ids_to_tokens(ids[:3])
check("prompt starts with exactly one <bos>", first_tokens[0] == "<bos>" and first_tokens[1] != "<bos>", str(first_tokens))

# 3.2 prompt lengths
lengths = pd.Series(tok_llm.prompt_lengths(followups))
print(lengths.describe(percentiles=[0.5, 0.9, 0.95, 0.99]).round(0).to_string())
over_old = int((lengths > 1400).sum())
over_new = int((lengths > params.max_input_tokens).sum())
print(f"over 1400 (old limit): {over_old} ({over_old / len(lengths):.0%})")
check(f"no prompt exceeds max_input_tokens={params.max_input_tokens}", over_new == 0, f"{over_new} over")

# 3.3 batch plan
plan = tok_llm._make_batches(lengths.tolist())
capped = [min(x, params.max_input_tokens) for x in lengths]
within_budget = all(len(b) == 1 or len(b) * max(capped[i] for i in b) <= params.max_batch_tokens for b in plan)
covers_all = sorted(i for b in plan for i in b) == list(range(len(lengths)))
check("every batch within token budget", within_budget)
check("batch plan covers every row exactly once", covers_all)
sizes = pd.Series([len(b) for b in plan]).value_counts().sort_index().to_dict()
print(f"generate() calls: {len(plan)} with budget batching vs {math.ceil(len(lengths) / 2)} with fixed batches of 2")
print(f"batch sizes (size: count): {sizes}")

## 4. Old vs new scoring on a sample

- **Old**: double `<bos>`, fp16, `max_input_tokens=1400`, fixed batches of 2.
- **New**: production `SentimentParams` (single `<bos>`, bf16, 2560 tokens, token-budget batching).

The sample always includes the longest answers, so peak memory here is the worst case.
Each variant runs in its own process (`sentiment_test_utils.py`), so GPU memory is fully released
between them. The cell shows no progress bars while it runs; expect a few minutes.
After it runs, review `disagreements` by hand: which label is right?

In [ ]:
from sentiment_test_utils import run_in_subprocess, score_variant

SAMPLE_SIZE = 50
N_LONGEST = 6
SEED = 42

answer_len = followups["ask_sa_answer"].astype(str).str.len()
longest_idx = answer_len.sort_values(ascending=False).index[:N_LONGEST]
rest = followups.drop(index=longest_idx)
sample = pd.concat(
    [followups.loc[longest_idx], rest.sample(n=min(SAMPLE_SIZE - N_LONGEST, len(rest)), random_state=SEED)]
).reset_index(drop=True)
print(f"Sample: {len(sample)} of {len(followups)} follow-ups")

check_gpu_free("section 4")
runs = {}
for variant in ["old", "new"]:
    runs[variant] = run_in_subprocess(score_variant, variant, sample, params, token)
    r = runs[variant]
    print(
        f"{variant}: load {r['load_time']:.0f}s, classify {r['classify_time']:.0f}s, "
        f"model {r['after_load_gb']:.1f} GB, peak {r['peak_gb']:.1f} GB"
    )

old, new = runs["old"]["result"], runs["new"]["result"]
old_cut = pd.Series(runs["old"]["lengths"]) > runs["old"]["max_input_tokens"]
old_time, new_time = runs["old"]["classify_time"], runs["new"]["classify_time"]
old_peak, new_peak = runs["old"]["peak_gb"], runs["new"]["peak_gb"]

parent_gb = torch.cuda.memory_allocated() / 1024**3
check("kernel holds no model memory after section 4", parent_gb < 1, f"{parent_gb:.1f} GB")

In [ ]:
cmp = pd.DataFrame(
    {
        "followup_query": sample["followup_query"].astype(str).str.slice(0, 120),
        "old_cut": old_cut,
        "label_old": old["label"],
        "label_new": new["label"],
        "subtype_old": old["subtype"],
        "subtype_new": new["subtype"],
        "rationale_new": new["rationale"],
    }
)
same = cmp["label_old"] == cmp["label_new"]
cut = cmp["old_cut"]
unparsed_old = int(old["rationale"].isin(sj.PARSE_FAILED).sum())
unparsed_new = int(new["rationale"].isin(sj.PARSE_FAILED).sum())

summary = pd.Series(
    {
        "rows": len(cmp),
        "label agreement, all": f"{same.mean():.0%}",
        f"label agreement, cut before ({cut.sum()} rows)": f"{same[cut].mean():.0%}" if cut.any() else "-",
        f"label agreement, not cut ({(~cut).sum()} rows)": f"{same[~cut].mean():.0%}" if (~cut).any() else "-",
        "subtype agreement, all": f"{(cmp['subtype_old'] == cmp['subtype_new']).mean():.0%}",
        "unparsed old / new": f"{unparsed_old} / {unparsed_new}",
        "classify time old / new, s": f"{old_time:.0f} / {new_time:.0f}",
        "peak GPU old / new, GB": f"{old_peak:.1f} / {new_peak:.1f}",
    }
)
print(summary.to_string())

check("new: unparsed not worse than old", unparsed_new <= unparsed_old, f"{unparsed_old} -> {unparsed_new}")
check("new: peak GPU memory below 95% of the card", new_peak < 0.95 * GPU_TOTAL_GB, f"{new_peak:.1f} of {GPU_TOTAL_GB:.1f} GB")
# 10% + 5 s tolerance for timing noise
check("new: not slower than old", new_time <= old_time * 1.1 + 5, f"{old_time:.0f}s -> {new_time:.0f}s")

pd.crosstab(cmp["label_old"], cmp["label_new"])

In [ ]:
# Review by hand: which label fits the follow-up better?
disagreements = cmp[~same]
pd.set_option("display.max_colwidth", 120)
disagreements

In [ ]:
import accelerate
import sys
import torch
import transformers

print(sys.version)
print("torch", torch.__version__, "cuda", torch.version.cuda)
print("transformers", transformers.__version__)
print("accelerate", accelerate.__version__)

## 5. Full production scoring (all follow-ups)

Same call as the job (`score_followups` + `prepare_output`), only without the Redshift load.
Time includes model loading.

In [ ]:
free_gpu()
check_gpu_free("section 5")
start = time.time()
scored = sj.score_followups(followups, params, token)
full_time = time.time() - start
full_peak = peak_gb()
output = sj.prepare_output(scored)
free_gpu()

text_max = output[sj.TEXT_COLUMNS].apply(lambda s: s.str.len().max()).max()
allowed_labels = {"POSITIVE", "NEGATIVE", "CLARIFICATION", "NEUTRAL"}
unparsed = int(output["rationale"].isin(sj.PARSE_FAILED).sum())

print(f"rows: {len(output)}, time: {full_time / 60:.1f} min, peak GPU: {full_peak:.1f} GB")
check("one output row per follow-up", len(output) == len(followups), f"{len(output)} / {len(followups)}")
check("output columns match the table", list(output.columns) == sj.OUTPUT_COLUMNS)
check("labels are valid", output["label"].isin(allowed_labels).all())
check("text columns cut to 256 chars", text_max <= sj.TEXT_MAX_CHARS, f"max {text_max}")
check("user_id / chat_id are integers", all(pd.api.types.is_integer_dtype(output[c]) for c in ["user_id", "chat_id"]))
check("followup_date filled", output["followup_date"].notna().all())
check("unparsed rows below 5%", unparsed <= 0.05 * len(output), f"{unparsed} rows")
check("peak GPU memory below 95% of the card", full_peak < 0.95 * GPU_TOTAL_GB, f"{full_peak:.1f} GB")
output.head()

## 6. COPY into a scratch table

Creates `ds.tmp_ask_sa_sentiment_load_test` with the same structure as the production table,
loads `output` through the production `load_df_to_redshift`, reads it back and drops the table.
Needs CREATE rights in the `ds` schema.

In [ ]:
SCRATCH_TABLE = "ds.tmp_ask_sa_sentiment_load_test"


def run_sql(sql):
    with redshift_connect(settings) as conn:
        cur = conn.cursor()
        cur.execute(sql)
        conn.commit()


run_sql(f"DROP TABLE IF EXISTS {SCRATCH_TABLE}")
run_sql(f"CREATE TABLE {SCRATCH_TABLE} (LIKE {settings.sentiment_table})")
try:
    loaded = load_df_to_redshift(settings, output, SCRATCH_TABLE, sj.OUTPUT_COLUMNS, "result_load_test.csv")
    counted = fetch_df(settings, f"SELECT COUNT(*) AS n FROM {SCRATCH_TABLE}")["n"].iloc[0]
    check("COPY loaded every row", loaded == counted == len(output), f"loaded {loaded}, in table {counted}")
    back = fetch_df(settings, f"SELECT * FROM {SCRATCH_TABLE} ORDER BY chat_id, followup_number LIMIT 5")
finally:
    run_sql(f"DROP TABLE IF EXISTS {SCRATCH_TABLE}")
back

## Summary

In [ ]:
summary_df = pd.DataFrame(RESULTS, columns=["status", "test", "details"])
failed = summary_df[summary_df["status"] == "FAIL"]
print(f"{len(summary_df) - len(failed)} passed, {len(failed)} failed")
summary_df